# Payment Transaction Audit Tool
### Automated sample selection using Benford's Law and machine learning

---
**How to use:**
1. Place your Excel dataset file in the `data/` folder.
2. Update the filename in **Step 0** below. Also check if the number of samples to be selected is as desired.
3. Proceed to run each cell in order from **STEP 0 to STEP 5** (To run a cell, please click into the cell, and press Shift + Enter).
4. Find your outputs in the `output/` folder.
---

In [ ]:
# ============================================================
# STEP 0 — Configure
# Update INPUT_FILE with the name of your Excel file.
# Check if SAMPLE_SIZE is as desired.
# ============================================================

# Please delete 'placeholder' and replace with the name of your Excel file (Reminder: Your excel file must be placed in the data/ folder)
# DO NOT delete '.xlsx' because this identifies your file is an Excel.
INPUT_FILE = 'placeholder.xlsx'

# Number of vouchers to select as the audit sample (default: 25; If change is required, please change to any positive whole number)
SAMPLE_SIZE = 25

# ---------------------------------------------------------------
# Advanced settings — safe to leave as defaults
# ---------------------------------------------------------------
# Weights for composite risk score (must sum to 1.0)
WEIGHTS = {
    'if_score':          0.30,   # Isolation Forest
    'lof_score':         0.25,   # Local Outlier Factor
    'zscore_score':      0.25,   # Statistical z-score
    'rule_flags_score':  0.15,   # Rule-based flags
    'benford_score':     0.05,   # Benford's Law (weak signal)
}

print(f"Configuration set: file='{INPUT_FILE}', sample size={SAMPLE_SIZE}")

In [ ]:
# ============================================================
# STEP 1 — Install required packages
# Run this cell once. It will install all dependencies.
# ============================================================
import subprocess, sys

print('Installing packages... (this may take a minute on first run)')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✔ All packages installed successfully.')
else:
    print('Installation output:')
    print(result.stdout)
    print(result.stderr)

In [ ]:
# ============================================================
# STEP 2 — Load and preview data
# ============================================================
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from IPython.display import display, HTML
from src.data_loader import load_transactions, AMOUNT_COL

data_path = os.path.join('data', INPUT_FILE)
df_raw = load_transactions(data_path)

# Summary statistics
date_min = df_raw['Voucher Accounting Date'].min()
date_max = df_raw['Voucher Accounting Date'].max()
print()
print(f'  Voucher Period: {date_min.strftime("%d %b %Y") if pd.notna(date_min) else "N/A"} '
      f'to {date_max.strftime("%d %b %Y") if pd.notna(date_max) else "N/A"}')
print(f'  Amount range : SGD {df_raw[AMOUNT_COL].min():,.2f} – {df_raw[AMOUNT_COL].max():,.2f}')
print(f'  Total amount : SGD {df_raw[AMOUNT_COL].sum():,.2f}')
print(f'  Unique vendors: {df_raw["Vendor ID"].nunique():,}')
print()
print('Preview (first 5 rows):')
display(df_raw.head())

In [ ]:
# ============================================================
# STEP 3 — Run full analysis
# This cell runs all tests and scores every transaction.
# ============================================================
from src.feature_engineering import engineer_features
from src import benfords_law
from src.ml_models import run_ensemble
from src.sample_selector import select_samples, WEIGHTS as DEFAULT_WEIGHTS

# Apply custom weights if changed in Step 0
import src.sample_selector as _sel
_sel.WEIGHTS.update(WEIGHTS)

print('--- Feature Engineering ---')
df, ml_features = engineer_features(df_raw.copy())

print()
print("--- Benford's Law Analysis ---")
df, benford_summary, benford_stats = benfords_law.analyze(df)

print()
print('--- Machine Learning Models ---')
df = run_ensemble(df, ml_features)

print()
print('--- Sample Selection (line scoring → voucher rollup) ---')
df_scored, df_vouchers, selected_vouchers = select_samples(df, n_samples=SAMPLE_SIZE)

print()
print('Analysis complete.')

In [ ]:
# ============================================================
# STEP 4 — View selected vouchers
# ============================================================
from IPython.display import display
import pandas as pd

display_cols = [
    'Sample #', 'Voucher ID', 'Vendor Name', 'Invoice Number(s)',
    'voucher_line_count', 'voucher_score', 'voucher_risk_tier',
    'voucher_reason_codes', 'Sample_Rationale',
]
display_cols = [c for c in display_cols if c in selected_vouchers.columns]

sub = selected_vouchers[display_cols].copy()
if 'voucher_score' in sub.columns:
    sub['voucher_score'] = sub['voucher_score'].round(4)

def _row_colour(row):
    tier = row.get('voucher_risk_tier', 'LOW')
    colour = {'HIGH': '#FFB3B3', 'MEDIUM': '#FFDDB3', 'LOW': '#FFFAB3'}.get(tier, '#FFFAB3')
    return [f'background-color: {colour}'] * len(row)

styled = sub.style.apply(_row_colour, axis=1).set_properties(**{
    'font-size': '12px', 'text-align': 'left'
})
display(styled)

In [ ]:
# ============================================================
# STEP 5 — Generate output files
# Creates: Excel workbook, Word report
# ============================================================
import os
from src.excel_exporter import export_excel
from src.report_generator import export_word_report

os.makedirs('output', exist_ok=True)

excel_path = os.path.join('output', 'selected_samples.xlsx')
word_path  = os.path.join('output', 'audit_report.docx')

print('--- Generating Excel workbook ---')
export_excel(df_scored, df_vouchers, selected_vouchers,
             benford_summary, benford_stats, excel_path)

print()
print('--- Generating Word report ---')
export_word_report(df_scored, df_vouchers, selected_vouchers,
                   benford_stats, word_path)

print()
print('=' * 50)
print('OUTPUT FILES READY')
print('=' * 50)
print(f'  Excel  : {os.path.abspath(excel_path)}')
print(f'  Report : {os.path.abspath(word_path)}')
print('=' * 50)